In [1]:
import sqlite3
import pandas as pd
import plotly.express as px

In [2]:
## === Get DataFrame from database of a specific group ===
gname = 'Grupo Deus é Amor'  # target group name

conn = sqlite3.connect("dataset/railway.sqlite")
cursor = conn.cursor()

query = "\
    SELECT e.id AS event_id, e.name AS event_name, e.start_date_time AS event_time, \
        p.id AS participant_id, p.full_name AS participant_name, c.timestamp AS checkin_time, \
        p.birth_date AS participant_birth, p.gender AS participant_gender \
    FROM check_ins AS c \
    JOIN events AS e ON c.event_id = e.id \
    JOIN participants AS p ON c.participant_id = p.id \
    WHERE e.group_id = (SELECT id FROM groups WHERE name = ?)"
# cursor.execute(query, (gname,))
# results = cursor.fetchall()
df = pd.read_sql(query, conn, params=(gname,))

In [3]:
### === Data Cleaning and Preprocessing ===

## Check for missing values
# df.isnull().sum()
# df.isna().sum()

# Convert date columns to datetime format
for col in ["event_time", "checkin_time", "participant_birth"]:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    
## Add age column
df["participant_age"] = (pd.to_datetime("today") - df["participant_birth"]).dt.days // 365



In [4]:
# # abnormal age value check
# abnormal_age_ids = df[df["participant_age"] <= 7]["participant_id"].drop_duplicates().tolist()

# # Get contact info for abnormal age participants
# placeholders = ",".join("?" * len(abnormal_age_ids))
# query = f"SELECT id, full_name, email, phone, birth_date FROM participants WHERE id IN ({placeholders})"

# contact_info = pd.read_sql(query, conn, params=abnormal_age_ids)
# contact_info.to_csv("dataset/abnormal_age_participants.csv", index=False)

conn.close()

In [6]:
df.dtypes

event_id                      object
event_name                    object
event_time            datetime64[ns]
participant_id                object
participant_name              object
checkin_time          datetime64[ns]
participant_birth     datetime64[ns]
participant_gender            object
participant_age                int64
dtype: object

In [ ]:
# event_id, event_name, event_time, participant_id, participant_name, checkin_time, participant_birth, participant_gender

list

## Group Participants Analysis

In [20]:
# all registered participants in the group
participants = df[["participant_id", "participant_name", "participant_gender", "participant_age"]].drop_duplicates().reset_index(drop=True)

# getting rid of out of bound age participants
participants = participants[(participants["participant_age"] > 7) & (participants["participant_age"] < 30)].reset_index(drop=True)

# add attendance count
attendance = df.groupby("participant_id").size().reset_index(name="attendance_count")
participants = participants.merge(attendance, on="participant_id", how="left")

# rename columns for better readability
participants.columns = ["id", "name", "gender", "age", "attendance_count"]
participants.head()

,id,name,gender,age,attendance_count
0,59fa3bcd-4903-4349-ac44-4af3352efc93,Felipe Camargo,MALE,16,2
1,346c4a15-bf18-4917-9877-48df46d84fdc,Felipe Vieira Coelho,MALE,13,1
2,13762581-ab39-4b01-ab0b-274ac8c6bceb,Antônio Martini,MALE,17,2
3,4226528c-c1c4-4ed5-ad7f-539378323e31,Otávio Gazarini Silva,MALE,16,2
4,4cd326fe-1c19-414c-9740-094e5030f90c,Francisco Neri dos Santos Soares,MALE,16,2


In [87]:
participants.dtypes

participant_id        object
participant_name      object
participant_gender    object
participant_age        int64
attendance_count       int64
dtype: object

In [21]:
# Gender distribution
gender = participants["gender"]
gender_info = pd.DataFrame({'count': gender.value_counts(), 
                            'percentage': (gender.value_counts(normalize=True) * 100).round(2)})
gender_info.reset_index(inplace=True)
gender_info

,gender,count,percentage
0,FEMALE,312,55.61
1,MALE,249,44.39


In [22]:
# Age distribution
def get_age_info(age_data):
    print(f"Average age: {age_data.mean():.1f} \nMedian age: {age_data.median()} \nAge range: {age_data.min()} - {age_data.max()}")

overall_age = participants['age']
get_age_info(overall_age)
overall_age.value_counts().head()

Average age: 14.6 
Median age: 14.0 
Age range: 11 - 25


age
14    138
15    127
13    108
16     84
12     37
Name: count, dtype: int64

In [23]:
female_age = participants[participants["gender"] == "FEMALE"]["age"]
male_age = participants[participants["gender"] == "MALE"]["age"]
print("Female Participants Age Info:")
get_age_info(female_age)
print("\nMale Participants Age Info:")
get_age_info(male_age)


Female Participants Age Info:
Average age: 14.3 
Median age: 14.0 
Age range: 11 - 18

Male Participants Age Info:
Average age: 15.1 
Median age: 15.0 
Age range: 12 - 25


In [24]:
# histogram: age distribution by gender 
hist = px.histogram(participants,
             x='age',
             color='gender',
             barmode='group', # 'group' - side by side, 'stack' - stacked bars
             title='Age Distribution')
hist.update_layout(xaxis_title='Age', yaxis_title='Count')
hist.update_xaxes(dtick=1)
hist.show()


In [25]:
px.violin(participants, x="gender", y="age", 
          color="gender", box=True,
          title="Age Distribution by Gender")